# 🇮🇳 Sahayak — Fine-tuning the Risk Classifier (Day 3 ⭐)

**The differentiator, in one notebook.** We fine-tune **InLegalBERT**
(BERT pretrained on 5.4M Indian legal documents) with **LoRA** so that a
clause comes in and a *direction-of-favor* label comes out:

| label | meaning | asymmetry mapping (Day 4) |
|---|---|---|
| `balanced` (0) | roughly even obligations | ~0 |
| `favors_them` (1) | disproportionately protects the OTHER party | negative |
| `favors_you` (2) | disproportionately protects OUR reader | positive |

**Why LoRA?** Full fine-tuning updates every weight (110M) — too much for
a free T4. LoRA freezes the base model and trains tiny low-rank adapter
matrices (~0.5% of params). Same idea, ~200x fewer trained numbers.

**Honest caveats (read before bragging):** the seed set is 58 train / 15 val
clauses — demo-grade. Metrics will be optimistic-ish and wobbly; the upgrade
path is the Kaggle Indian-clauses dataset + CUAD mapping. The *pipeline* is
the point today; scale comes later.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then run cells
top to bottom. You'll upload two JSONL files and paste one HF token.

## 1 — Install the ML stack

In [ ]:
%pip install -q "transformers>=4.46" datasets peft accelerate scikit-learn
print("✅ installed")

## 2 — Imports + GPU check

In [ ]:
import json, random, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

set_seed(42)
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU, then rerun from cell 1."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "law-ai/InLegalBERT"   # 🇮🇳 Indian legal BERT (SC + HC judgments)
LABELS = ["balanced", "favors_them", "favors_you"]  # fixed id order 0/1/2
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

## 3 — Upload the dataset

Select **`train.jsonl` AND `val.jsonl` together** in the file picker
(they live in the repo under `model/data/`).

In [ ]:
from google.colab import files
print("Pick train.jsonl AND val.jsonl (Ctrl/Cmd-click both):")
up = files.upload()

def read_jsonl(name):
    rows = [json.loads(l) for l in up[name].decode("utf-8").splitlines() if l.strip()]
    return rows

train_rows = read_jsonl("train.jsonl")
val_rows   = read_jsonl("val.jsonl")
print(f"train={len(train_rows)}  val={len(val_rows)}")
assert train_rows and val_rows, "Upload both train.jsonl and val.jsonl."

ds = {}
for split, rows in (("train", train_rows), ("val", val_rows)):
    ds[split] = Dataset.from_list(
        [{"text": r["text"], "label": label2id[r["label"]]} for r in rows]
    )
print(ds["train"][0])

## 4 — Tokenize

BERT eats tokens, not sentences. Clauses are short — 256 subword tokens
covers nearly all of them with headroom.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

ds = {k: v.map(tokenize, batched=True) for k, v in ds.items()}
print(ds["train"][0]["input_ids"][:12], "...")

## 5 — InLegalBERT + LoRA adapters

`r` = adapter rank (capacity), `lora_alpha` = scaling. `query`/`value` are
the attention projections LoRA pokes holes in — the standard BERT recipe.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

lora = LoraConfig(
    task_type="SEQ_CLS",
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["query", "value"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # expect ~0.5-1% trainable

## 6 — Train

5 epochs on 58 examples is minutes on a T4. Eval every epoch, keep the best
checkpoint by macro-F1 (treats all 3 classes equally, unlike raw accuracy).

In [ ]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="sahayak-risk-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds["train"], eval_dataset=ds["val"],
                  compute_metrics=compute_metrics)
trainer.train()

## 7 — Evaluate honestly

Accuracy alone can lie (3 classes → 33% by chance). We want the
**confusion matrix**: which direction does the model *confuse*?
Confusing `favors_them` ↔ `favors_you` is the costly error — it would
flip an asymmetry score's sign.

In [ ]:
pred = trainer.predict(ds["val"])
y_true = pred.label_ids
y_hat  = pred.predictions.argmax(-1)

print(classification_report(y_true, y_hat, target_names=LABELS, digits=3))
print("Confusion matrix (rows=true, cols=predicted):")
print("        " + "  ".join(f"{l:>11}" for l in LABELS))
for name, row in zip(LABELS, confusion_matrix(y_true, y_hat)):
    print(f"{name:>10} " + "  ".join(f"{v:>11}" for v in row))

## 8 — Sanity check on clauses it has NEVER seen

Three fresh clauses, one per class. If these land right, the loop works
and Day 3 is a go — even with modest val metrics.

In [ ]:
import numpy as np

tests = [
    "The Company may terminate this Agreement at any time, for any reason, without notice or compensation.",  # favors_them
    "Either party may terminate this Agreement with thirty days' prior written notice to the other.",          # balanced
    "All overdue payments shall accrue interest at two percent per month, automatically and without notice.",  # favors_you
]
enc = tokenizer(tests, truncation=True, padding=True, max_length=256, return_tensors="pt").to(model.device)
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, -1).cpu().numpy()

for text, p in zip(tests, probs):
    top = p.argmax()
    print(f"{p[top]:.0%} {LABELS[top]:>11} | {text[:70]}...")
    print("        " + "  ".join(f"{l}={v:.2f}" for l, v in zip(LABELS, p)))

## 9 — Push YOUR model to the Hugging Face Hub

Creates a free account token at https://huggingface.co/settings/tokens
(type: **Write**) — paste it when prompted. This cell publishes the LoRA
adapter + tokenizer to your own repo.

In [ ]:
from huggingface_hub import notebook_login, whoami
notebook_login()  # paste a Write-scoped token from huggingface.co/settings/tokens

repo_id = f"{whoami()['name']}/sahayak-risk-classifier"
model.push_to_hub(repo_id)      # LoRA adapter only — small (~few MB)
tokenizer.push_to_hub(repo_id)
print("✅ pushed:", repo_id)

## 10 — Done ✅ → hand the repo id to the pipeline

Copy the `repo_id` printed above (looks like `yourname/sahayak-risk-classifier`)
and tell ZCode — the Risk node (Day 4.1) loads it with:

```python
from peft import AutoPeftModelForSequenceClassification
model = AutoPeftModelForSequenceClassification.from_pretrained(repo_id)
tok   = AutoTokenizer.from_pretrained(repo_id)
```

**Upgrade path (post-lock-in):** merge Kaggle Indian-clauses data + CUAD
mapping, retrain, compare macro-F1. The adapter repo makes retraining
cheap — LoRA keeps the big base frozen.